# SmolLM2-135M to Sparse MoE: Upcycling, Training & Evaluation

This notebook runs the complete pipeline:
1. Mount Google Drive for persistent checkpoints and logs.
2. Curate and shard multi-domain contextual datasets.
3. Run a fast pre-flight sanity check.
4. Launch fault-tolerant dual-worker Sparse MoE training on GPU.
5. Normalize logs and export final model weights directly to Google Drive.
6. Run interactive inference.

## 1. Google Drive Mount & Hardware Verification

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!nvidia-smi


## 2. Setup Repository & Dependencies

In [ ]:
import os
if not os.path.exists("/content/linear_MoE_upcycling"):
    !git clone https://github.com/thinktinkergreen/linear_MoE_upcycling.git /content/linear_MoE_upcycling
else:
    !git -C /content/linear_MoE_upcycling pull origin main

%cd /content/linear_MoE_upcycling
!pip install -q -r requirements.txt


## 3. Curate and Shard the Multi-Domain Dataset

In [ ]:
!python src/data/curate.py \n  --output_dir /content/drive/MyDrive/linear_MoE_upcycling \n  --num_shards 2


## 4. Pre-Flight Sanity Check (Verifies backprop & zero NaNs in 15 seconds)

In [ ]:
!python scripts/sanity_check.py


## 5. Launch Fault-Tolerant Dual-Worker Training

In [ ]:
!python scripts/run_train.py --config configs/default_config.yaml


## 6. Normalize and Sync Training Logs

In [ ]:
# Normalize logged metrics and copy to repository
!python scripts/rescale_logs.py --log_dir /content/drive/MyDrive/linear_MoE_upcycling
!mkdir -p /content/linear_MoE_upcycling/logs/
!cp /content/drive/MyDrive/linear_MoE_upcycling/*.csv /content/linear_MoE_upcycling/logs/
!ls -lh /content/linear_MoE_upcycling/logs/


## 7. Export Fine-Tuned MoE Model to Google Drive

In [ ]:
# Saves model weights, config, and tokenizer directly into Google Drive
!python scripts/save_final_model.py --config configs/default_config.yaml
!ls -lh /content/drive/MyDrive/linear_MoE_upcycling/final_moe_model/


## 8. Interactive Evaluation & Prompt Generation

In [ ]:
!python scripts/evaluate_moe.py \n  --config configs/default_config.yaml \n  --prompt "What are the main kharif crops grown in Maharashtra and what soil do they need?"
